# CelebA-HQ Face Generation — Evaluation

End-to-end FID evaluation for one DDPM training run produced by
`notebooks/colab_celebahq_train.ipynb`.

Pipeline:
1. Mount Drive and clone the repo.
2. Read the training `run_config.json` so the eval config (image_size, timesteps,
   base_channels, time_dim) cannot drift from training.
3. Generate **300 face images** from the trained checkpoint.
4. Export the **300 reserved CelebA-HQ test images** as PNGs.
5. Compute **FID** between the two folders.
6. Save real/fake grids and a `fid.json` summary into the run directory.
7. Aggregate FID across all runs into a single table for the report.

**To evaluate a different run**, change `RUN_NAME` in the config cell and re-run all cells.

## 1 · Mount Drive and clone repo

In [ ]:
import os
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/AAnanya19/Human-Faces-Generation-Diffusion-Models.git"
BRANCH   = "feat/eval"
WORKDIR  = Path("/content/Human-Faces-Generation-Diffusion-Models")

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)

!git clone --branch "$BRANCH" "$REPO_URL" "$WORKDIR"

os.chdir(WORKDIR)
print("Project root:", WORKDIR)

## 2 · Install dependencies

In [ ]:
assert os.path.isfile("requirements.txt"), "Run the clone cell first."
!pip install -q -r requirements.txt
!pip install -q torch-fidelity

## 3 · Run config

`RUN_NAME` is the **only** value you usually need to change. Everything else
(image size, U-Net width, number of timesteps, etc.) is read from the
training run's `run_config.json` so the eval cannot drift from training.

In [ ]:
import json
from pathlib import Path

DRIVE_ROOT       = Path("/content/drive/MyDrive/aml")
RUNS_ROOT        = DRIVE_ROOT / "ddpm_runs"
RUN_NAME         = "celebahq_run_001"   # ← edit this to evaluate a different run
RUN_DIR          = RUNS_ROOT / RUN_NAME

CHECKPOINT_PATH  = RUN_DIR / "ddpm_final.pth"
TEST_FILES_PATH  = RUN_DIR / "test_files.txt"
RUN_CONFIG_PATH  = RUN_DIR / "run_config.json"

REAL_DIR         = RUN_DIR / "real_test_set"
FAKE_DIR         = RUN_DIR / "generated_faces"
EVAL_DIR         = RUN_DIR / "eval"
FID_JSON_PATH    = RUN_DIR / "fid.json"

for required in (CHECKPOINT_PATH, TEST_FILES_PATH, RUN_CONFIG_PATH):
    if not required.is_file():
        raise FileNotFoundError(
            f"Missing artefact from training run: {required}\n"
            f"Make sure RUN_NAME is correct and that training finished and uploaded outputs."
        )

run_config = json.loads(RUN_CONFIG_PATH.read_text())
model_cfg  = run_config.get("model", {})

IMAGE_SIZE     = int(run_config.get("image_size", 256))
TIMESTEPS      = int(run_config.get("timesteps", 1000))
BASE_CHANNELS  = int(model_cfg.get("base_channels", 128))
TIME_DIM       = int(model_cfg.get("time_dim", 512))
BATCH_SIZE     = 16   # generation batch; reduce if OOM
NUM_IMAGES     = 300  # required by the project spec

print("RUN_NAME      :", RUN_NAME)
print("image_size    :", IMAGE_SIZE)
print("timesteps     :", TIMESTEPS)
print("base_channels :", BASE_CHANNELS)
print("time_dim      :", TIME_DIM)
print("checkpoint    :", CHECKPOINT_PATH)

## 4 · Verify GPU

In [ ]:
import torch

print("torch  :", torch.__version__)
print("device :", "cuda" if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not active.\n"
        "Go to Runtime → Change runtime type → T4 GPU (or A100 if available)."
    )
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

## 5 · Generate 300 face images

Calls `scripts/generate_faces.py` with the same model hyperparameters used during
training. Individual PNGs and a preview grid are written into `generated_faces/`.

In [ ]:
FAKE_DIR.mkdir(parents=True, exist_ok=True)

!python3 scripts/generate_faces.py \
    --checkpoint    "$CHECKPOINT_PATH" \
    --out_dir       "$FAKE_DIR" \
    --num_images    $NUM_IMAGES \
    --batch_size    $BATCH_SIZE \
    --image_size    $IMAGE_SIZE \
    --timesteps     $TIMESTEPS \
    --base_channels $BASE_CHANNELS \
    --time_dim      $TIME_DIM \
    --device        cuda

## 6 · Export the 300 real test images

Reads `test_files.txt` from the training run and writes resized PNGs into
`real_test_set/`. This is the **same 300 images** that were held out from
training, satisfying the project spec.

In [ ]:
REAL_DIR.mkdir(parents=True, exist_ok=True)

!python3 scripts/export_celebahq_reference.py \
    --test-files "$TEST_FILES_PATH" \
    --out-dir    "$REAL_DIR" \
    --image-size $IMAGE_SIZE

## 7 · Compute FID and save grids

In [ ]:
EVAL_DIR.mkdir(parents=True, exist_ok=True)
GEN_GRID  = EVAL_DIR / "generated_grid.png"
REAL_GRID = EVAL_DIR / "real_grid.png"

!python3 scripts/evaluate_fid.py \
    --real-dir       "$REAL_DIR" \
    --fake-dir       "$FAKE_DIR" \
    --image-size     $IMAGE_SIZE \
    --run-name       "$RUN_NAME" \
    --out-json       "$FID_JSON_PATH" \
    --grid-out       "$GEN_GRID" \
    --real-grid-out  "$REAL_GRID"

## 8 · Show the saved grids and FID

In [ ]:
from IPython.display import Image, display, Markdown

fid_payload = json.loads(FID_JSON_PATH.read_text())
display(Markdown(f"### FID for `{RUN_NAME}`: **{fid_payload['fid']:.4f}**"))
display(Markdown(f"_n_real={fid_payload['n_real']}, n_fake={fid_payload['n_fake']}, image_size={fid_payload['image_size']}_"))

if GEN_GRID.is_file():
    display(Markdown("#### Generated grid"))
    display(Image(filename=str(GEN_GRID)))
if REAL_GRID.is_file():
    display(Markdown("#### Real grid"))
    display(Image(filename=str(REAL_GRID)))

## 9 · Plot training loss curve

Quick figure for the report; reads the per-epoch averages logged by training.

In [ ]:
import csv
import matplotlib.pyplot as plt

loss_log = RUN_DIR / "loss_log.csv"
loss_plot = EVAL_DIR / "loss_curve.png"

if loss_log.is_file():
    epochs, losses = [], []
    with loss_log.open() as f:
        reader = csv.DictReader(f)
        for row in reader:
            epochs.append(int(row["epoch"]))
            losses.append(float(row["avg_loss"]))
    plt.figure(figsize=(6, 3.5))
    plt.plot(epochs, losses)
    plt.xlabel("Epoch")
    plt.ylabel("Average MSE noise loss")
    plt.title(f"Training loss — {RUN_NAME}")
    plt.tight_layout()
    plt.savefig(loss_plot, dpi=150)
    plt.show()
    print("Saved loss curve:", loss_plot)
else:
    print("No loss_log.csv found in run directory; skipping loss curve.")

## 10 · Aggregate FID across all runs

Joins every run's `run_config.json` and `fid.json` into a single table
(`results/celebahq/eval/fid_summary.{csv,md}`) for the report.

In [ ]:
SUMMARY_DIR = Path("results/celebahq/eval")
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_CSV = SUMMARY_DIR / "fid_summary.csv"
SUMMARY_MD  = SUMMARY_DIR / "fid_summary.md"

!python3 scripts/aggregate_fid.py \
    --runs-dir "$RUNS_ROOT" \
    --out-csv  "$SUMMARY_CSV" \
    --out-md   "$SUMMARY_MD" \
    --require-fid

from IPython.display import Markdown, display
if SUMMARY_MD.is_file():
    display(Markdown(SUMMARY_MD.read_text()))

---
**Outputs of this notebook (in `MyDrive/aml/ddpm_runs/<RUN_NAME>/`):**
- `generated_faces/` — 300 PNGs from the trained model
- `real_test_set/`   — 300 PNGs from the held-out test split
- `eval/generated_grid.png`, `eval/real_grid.png`, `eval/loss_curve.png`
- `fid.json` — FID number plus paths and image counts

Plus repo-level: `results/celebahq/eval/fid_summary.{csv,md}` aggregating every run.